# Exploratory Market Analysis: Maharashtra Vehicle Registrations (2017–2026)

This notebook conducts the exploratory time-series and market share analysis for Maharashtra statewide registrations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose

# Load the standardized Maharashtra monthly analytical dataset
df = pd.read_csv('data/clean/registrations_maharashtra_monthly.csv', dtype={'month_id': str})
df['date'] = pd.to_datetime(df['date'])
df.head()

## 1. Monthly Registration Trends & Summary Statistics

In [ ]:
monthly_tot = df.groupby(['month_id', 'date', 'reporting_window'], as_index=False)['count'].sum().sort_values('date')
monthly_tot['yoy_growth'] = monthly_tot['count'].pct_change(12) * 100

print('Descriptive Statistics across Windows:')
display_stats = pd.DataFrame({
    'Full Series (2017-26)': monthly_tot['count'].describe(),
    'Historical Training (2017-18)': monthly_tot[monthly_tot['reporting_window']=='historical_training']['count'].describe(),
    'Primary Window (2019-26)': monthly_tot[monthly_tot['reporting_window']=='primary_reporting']['count'].describe()
})
display_stats.round(1)

## 2. Fuel-Group Volumes & Market Share Transitions

In [ ]:
# Categorize into macro fuel groups
def categorize_macro(fg):
    if fg == 'Petrol': return 'Petrol (Standard)'
    elif fg == 'Petrol-E20': return 'Petrol-E20'
    elif fg in ['Petrol-CNG', 'Petrol-E20-CNG', 'CNG']: return 'CNG & Bi-Fuel'
    elif fg == 'Diesel': return 'Diesel'
    elif fg in ['Electric-BOV', 'Pure EV']: return 'Electric (Pure EV + BOV)'
    elif fg in ['Strong Hybrid EV', 'Petrol-Hybrid', 'Petrol-E20-Hybrid', 'PHEV']: return 'Hybrid'
    else: return 'Other'

df['macro_fuel'] = df['fuel_group'].apply(categorize_macro)
fuel_piv = df.pivot_table(index='date', columns='macro_fuel', values='count', aggfunc='sum', fill_value=0)
fuel_share = fuel_piv.div(fuel_piv.sum(axis=1), axis=0) * 100

# Plot fuel share over time
plt.figure(figsize=(14, 6))
for col in fuel_share.columns:
    plt.plot(fuel_share.index, fuel_share[col], label=col, lw=2)
plt.title('Maharashtra Monthly Fuel-Group Market Shares (2017–2026)', fontsize=13, fontweight='bold')
plt.ylabel('Market Share (%)')
plt.grid(True, alpha=0.3)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 3. E20 S-Curve Penetration Trajectory

In [ ]:
petrol_pure = fuel_piv['Petrol (Standard)'] + fuel_piv['Petrol-E20']
e20_pct = (fuel_piv['Petrol-E20'] / petrol_pure) * 100

plt.figure(figsize=(12, 5))
plt.plot(e20_pct.index, e20_pct, 'b-o', lw=2)
plt.axvline(pd.to_datetime('2023-02-01'), color='green', linestyle='--', label='E20 Rollout Pilot (Feb 2023)')
plt.axvline(pd.to_datetime('2023-04-01'), color='red', linestyle='--', label='BS-VI Phase 2 Mandate (Apr 2023)')
plt.title('E20 Penetration within Pure Petrol Fleet (%)', fontsize=13, fontweight='bold')
plt.ylabel('E20 Share of Petrol (%)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 4. Stationarity Diagnostics (ADF & KPSS)

In [ ]:
series = monthly_tot.set_index('date')['count']
adf_res = adfuller(series, autolag='AIC')
kpss_res = kpss(series, regression='c', nlags='auto')
diff_series = series.diff().dropna()
adf_diff = adfuller(diff_series, autolag='AIC')
kpss_diff = kpss(diff_series, regression='c', nlags='auto')

pd.DataFrame([
    {'Test': 'ADF Level', 'Stat': adf_res[0], 'p-value': adf_res[1], 'Stationary': adf_res[1]<0.05},
    {'Test': 'KPSS Level', 'Stat': kpss_res[0], 'p-value': kpss_res[1], 'Stationary': kpss_res[1]>=0.05},
    {'Test': 'ADF 1st Diff', 'Stat': adf_diff[0], 'p-value': adf_diff[1], 'Stationary': adf_diff[1]<0.05},
    {'Test': 'KPSS 1st Diff', 'Stat': kpss_diff[0], 'p-value': kpss_diff[1], 'Stationary': kpss_diff[1]>=0.05}
]).round(4)

## 5. Seasonal Decomposition & Structural Breaks

In [ ]:
decomp = seasonal_decompose(series, model='additive', period=12)
fig = decomp.plot()
fig.set_size_inches(14, 8)
plt.tight_layout()
plt.show()